# Project Overview
This project builds a semantic search engine from scratch using a Wikipedia dump. It involves data extraction, preprocessing, triplet mining for contrastive learning, training a custom embedding model, embedding text, indexing with FAISS, querying, reranking, and evaluation.

---

## Runtime Environment

All code is executed inside a Docker container using the following image:

```
nvcr.io/nvidia/tensorflow:23.12-tf2-py3
```

In [1]:
import json
import numpy as np
import os
import pprint
import sys
import tensorflow as tf
import zipfile
from sentence_transformers import SentenceTransformer

# Add root directory (one level up from notebooks/)
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(project_root)

2025-11-01 22:33:11.358982: I tensorflow/core/util/port.cc:111] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-01 22:33:11.381468: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9360] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-11-01 22:33:11.381503: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-11-01 22:33:11.381520: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1537] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-11-01 22:33:11.387364: I tensorflow/core/platform/cpu_feature_g

In [2]:
# Raw Wikipedia Data Paths
WIKIDATA_DIR = r'../data/raw/extracted_wikidata'
PAGE_SQL_PATH = r"../data/raw/enwiki-latest-page.sql"
PAGELINKS_SQL_PATH = r"../data/raw/enwiki-latest-pagelinks.sql"
LINKTARGET_SQL_PATH = r"../data/raw/enwiki-latest-linktarget.sql"

# Preprocessed Wikipedia Data Paths
WIKIDATA_JSON_DIR = r'../data/processed/wikidata_json'
WIKIDATA_JSONL_DIR = r"../data/processed/wikidata_jsonl"
ARTICLE_TITLES_JSON = r"../data/processed/article_titles.json"
ARTICLE_METADATA_OUTPUT_PATH = r"../data/custom_model/article_metadata.json"
WIKI_LINK_GRAPH_JSONL_PATH = r"../data/processed/wiki_link_graph_jsonl"
WIKI_LINK_GRAPH_DB_PATH = r"../data/processed/wiki_link_graph.db"

# Triplet Training Data
TRIPLETS_DIR = r"../data/processed/triplets/parallel_parts"

# Custom Model Training Paths
ARTICLE_METADATA_PATH = r"../data/custom_model/article_metadata.json"
VECTORIZER_DIR = r"../data/custom_model/saved_vectorizer"
WEIGHTS_DIR = r"../data/custom_model"
UNIQUE_ANCHORS_PATH = r"../data/custom_model/embeddings_output/unique_anchors.txt"

# Embedding and Indexing
EMBEDDINGS_DIR = r"../data/custom_model/embeddings_output"
EMBEDDINGS_CHUNK_PATH = r"../data/custom_model/embeddings_output/chunks"
FAISS_MODEL_PATH = r"../data/custom_model/faiss/faiss_index.index"
FAISS_PARA_INDEX_PATH = r"../data/processed/faiss_index/paragraphs.index"
FAISS_PARA_META_PATH = FAISS_PARA_INDEX_PATH + r".meta.json"

# Evaluation
TEST_QUERIES_PATH = r"../data/test_data/test_queries.json"

MODEL_NAME = "all-MiniLM-L6-v2"


---

## Step 1: Install WikiExtractor

Downloads and installs the `wikiextractor` tool from GitHub if not already present.  
Used to extract clean text from the raw Wikipedia XML dump.


In [10]:
if not os.path.isdir(r"../wikiextractor-master"):
    # Step 1: Download the ZIP file
    !curl -L -o ../wikiextractor.zip https://github.com/attardi/wikiextractor/archive/refs/heads/master.zip

    # Step 2: Extract it
    with zipfile.ZipFile(r"../wikiextractor.zip", 'r') as zip_ref:
        zip_ref.extractall(r"../")

    # Step 3: Delete the ZIP file
    os.remove(r"../wikiextractor.zip")

    # Step 4: Install Wikiextractor
    !pip install -e ../wikiextractor-master
else:
    print("Wikiextractor already exists")

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 49444    0 49444    0     0  61706      0 --:--:-- --:--:-- --:--:-- 61706
Obtaining file:///app/files/semantic-wiki-search/wikiextractor-master
  Preparing metadata (setup.py) ... done
  Running setup.py develop for wikiextractor


---

## Step 2: Download Wikipedia Dump

Downloads the full English Wikipedia dump (~20GB compressed) if not already downloaded.  
The file is used as input for WikiExtractor.

In [4]:
os.makedirs("../data/raw", exist_ok=True)
if not os.path.isfile(r"../data/raw/enwiki-latest-pages-articles.xml.bz2"):
    !wget -P ../data/raw https://dumps.wikimedia.org/enwiki/latest/enwiki-latest-pages-articles.xml.bz2
else:
    print("Wikipedia dump already downloaded")

--2025-06-26 23:36:09--  https://dumps.wikimedia.org/enwiki/latest/enwiki-latest-pages-articles.xml.bz2
Resolving dumps.wikimedia.org (dumps.wikimedia.org)... 208.80.154.71, 2620:0:861:3:208:80:154:71
Connecting to dumps.wikimedia.org (dumps.wikimedia.org)|208.80.154.71|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 24073395782 (22G) [application/octet-stream]
Saving to: ‘../data/raw/enwiki-latest-pages-articles.xml.bz2’

enwiki-latest-pages 100%[===================>]  22.42G  3.70MB/s    in 1h 40m  

2025-06-27 01:07:22 (3.81 MB/s) - ‘../data/raw/enwiki-latest-pages-articles.xml.bz2’ saved [24073395782/24073395782]



---

## Step 3: Extract XML Data

Runs WikiExtractor on the downloaded dump to extract raw XML data.  
Outputs a directory of raw cleaned articles in XML format.

In [11]:
if not os.path.isdir(r"../data/raw/extracted_wikidata"):
    !python -m wikiextractor.WikiExtractor ../data/raw/enwiki-latest-pages-articles.xml.bz2 -o ../data/raw/extracted_wikidata --no-templates
else:
    print("Wikipedia XML extract already exists")

INFO: Starting page extraction from ../data/raw/enwiki-latest-pages-articles.xml.bz2.
INFO: Using 11 extract processes.
INFO: Extracted 100000 articles (1220.0 art/s)
INFO: Extracted 200000 articles (1661.6 art/s)
INFO: Extracted 300000 articles (1950.8 art/s)
INFO: Extracted 400000 articles (2694.2 art/s)
INFO: Extracted 500000 articles (2583.7 art/s)
INFO: Extracted 600000 articles (2134.3 art/s)
INFO: Extracted 700000 articles (2553.3 art/s)
INFO: Extracted 800000 articles (3020.9 art/s)
INFO: Extracted 900000 articles (3329.2 art/s)
INFO: Extracted 1000000 articles (3383.9 art/s)
INFO: Extracted 1100000 articles (3568.7 art/s)
INFO: Extracted 1200000 articles (3785.6 art/s)
INFO: Extracted 1300000 articles (3657.0 art/s)
INFO: Extracted 1400000 articles (3868.3 art/s)
INFO: Extracted 1500000 articles (3931.0 art/s)
INFO: Extracted 1600000 articles (4108.3 art/s)
INFO: Extracted 1700000 articles (4142.6 art/s)
INFO: Extracted 1800000 articles (4147.7 art/s)
INFO: Extracted 1900000 a

---

## Step 4: Convert Extracted XML to JSON

Converts the raw XML files into structured JSON using `traverse_directory` from `data_prep.py`.  
Only keeps articles with 100+ words and cleans text formatting.

In [12]:
from utils.data_prep import traverse_directory, convert_json_array_to_jsonl

if not os.path.isdir(WIKIDATA_JSON_DIR):
    traverse_directory(WIKIDATA_DIR, WIKIDATA_JSON_DIR)
    convert_json_array_to_jsonl(WIKIDATA_JSON_DIR, WIKIDATA_JSONL_DIR)
else:
    print("wikidata_json already exists")

Converting JSON to JSONL: 100%|██████████| 19080/19080 [14:39<00:00, 21.69file/s] 


---

## Step 5: Download SQL Link Graph Files

Downloads required SQL dump files:
- `page.sql`
- `pagelinks.sql`
- `linktarget.sql`

These files are used to construct a graph of internal Wikipedia links.

In [13]:
if not os.path.isfile(r"../data/raw/enwiki-latest-page.sql"):
    !wget -P ../data/raw https://dumps.wikimedia.org/enwiki/latest/enwiki-latest-page.sql.gz
    !gunzip ../data/raw/enwiki-latest-page.sql.gz
else:
    print("Wikipedia pages already downloaded")

--2025-06-27 05:42:37--  https://dumps.wikimedia.org/enwiki/latest/enwiki-latest-page.sql.gz
Resolving dumps.wikimedia.org (dumps.wikimedia.org)... 208.80.154.71, 208.80.154.71, 2620:0:861:3:208:80:154:71
Connecting to dumps.wikimedia.org (dumps.wikimedia.org)|208.80.154.71|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2304622174 (2.1G) [application/octet-stream]
Saving to: ‘../data/raw/enwiki-latest-page.sql.gz’

enwiki-latest-page. 100%[===================>]   2.15G  3.74MB/s    in 9m 46s  

2025-06-27 05:51:33 (3.75 MB/s) - ‘../data/raw/enwiki-latest-page.sql.gz’ saved [2304622174/2304622174]



In [14]:
if not os.path.isfile(r"../data/raw/enwiki-latest-pagelinks.sql"):
    !wget -P ../data/raw https://dumps.wikimedia.org/enwiki/latest/enwiki-latest-pagelinks.sql.gz
    !gunzip ../data/raw/enwiki-latest-pagelinks.sql.gz
else:
    print("Wikipedia pagelinks already downloaded")

--2025-06-27 05:52:51--  https://dumps.wikimedia.org/enwiki/latest/enwiki-latest-pagelinks.sql.gz
Resolving dumps.wikimedia.org (dumps.wikimedia.org)... 208.80.154.71, 208.80.154.71, 2620:0:861:3:208:80:154:71
Connecting to dumps.wikimedia.org (dumps.wikimedia.org)|208.80.154.71|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6899634850 (6.4G) [application/octet-stream]
Saving to: ‘../data/raw/enwiki-latest-pagelinks.sql.gz’

enwiki-latest-pagel 100%[===================>]   6.43G  3.61MB/s    in 29m 53s 

2025-06-27 06:20:00 (3.67 MB/s) - ‘../data/raw/enwiki-latest-pagelinks.sql.gz’ saved [6899634850/6899634850]



In [15]:
if not os.path.isfile(r"../data/raw/enwiki-latest-linktarget.sql"):
    !wget -P ../data/raw https://dumps.wikimedia.org/enwiki/latest/enwiki-latest-linktarget.sql.gz
    !gunzip ../data/raw/enwiki-latest-linktarget.sql.gz
else:
    print("Wikipedia linktarget already downloaded")

--2025-06-27 06:25:15--  https://dumps.wikimedia.org/enwiki/latest/enwiki-latest-linktarget.sql.gz
Resolving dumps.wikimedia.org (dumps.wikimedia.org)... 208.80.154.71, 208.80.154.71, 2620:0:861:3:208:80:154:71
Connecting to dumps.wikimedia.org (dumps.wikimedia.org)|208.80.154.71|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1153954433 (1.1G) [application/octet-stream]
Saving to: ‘../data/raw/enwiki-latest-linktarget.sql.gz’

enwiki-latest-linkt 100%[===================>]   1.07G  3.11MB/s    in 5m 28s  

2025-06-27 06:30:14 (3.35 MB/s) - ‘../data/raw/enwiki-latest-linktarget.sql.gz’ saved [1153954433/1153954433]



---

## Step 6: Build Link Graph in SQLite

Parses the SQL files and constructs a JSONL-based link graph (title → [linked titles]),  
then loads it into a local SQLite database using `build_linkgraph_sqlite`.

In [5]:
from utils.sqlite_lookup import build_linkgraph_sqlite

build_linkgraph_sqlite(WIKI_LINK_GRAPH_JSONL_PATH, WIKI_LINK_GRAPH_DB_PATH)

In [4]:
from utils.link_graph import export_link_graph_to_jsonl

export_link_graph_to_jsonl(PAGE_SQL_PATH, PAGELINKS_SQL_PATH, LINKTARGET_SQL_PATH, WIKI_LINK_GRAPH_JSONL_PATH)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/06/28 04:31:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/06/28 04:31:48 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in standalone/kubernetes and LOCAL_DIRS in YARN).


## Step 7: Generate Metadata and Triplets via PySpark

- Initializes Spark  
- Loads the Wikipedia JSON dataset  
- Extracts and saves article and paragraph metadata  
- Randomly generates triplet data (anchor, positive, negative)  
  for training using `create_random_triplets` in `spark_functions.py`

In [3]:
from pyspark.sql import SparkSession
from utils.spark_functions import create_article_metadata

spark = SparkSession.builder \
        .appName("Capstone") \
        .master("local[*]") \
        .config("spark.driver.memory", "20g") \
        .config("spark.sql.shuffle.partitions", "100") \
        .config("spark.local.dir", "../spark-temp") \
        .config("spark.driver.maxResultSize", "2g") \
        .getOrCreate()

# Load entire directory (Spark will recursively find JSON files)
input_df = spark.read.option("multiLine", True) \
                     .option("recursiveFileLookup", "true") \
                     .json("../data/processed/wikidata_json")

# Generate article metadata
create_article_metadata(input_df, ARTICLE_METADATA_OUTPUT_PATH)

spark.stop()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/01 22:22:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/11/01 22:22:25 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in standalone/kubernetes and LOCAL_DIRS in YARN).
ERROR:root:Exception while sending command.==>                  (88 + 12) / 133]
Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
RuntimeError: reentrant call inside <_io.BufferedReader name=71>

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/loca

Py4JError: An error occurred while calling o39.json

---

## Step 8: Mine Hard Triplets Using FAISS + SQLite

- Loads a prebuilt FAISS index  
- Retrieves linked paragraphs via SQLite  
- Uses multiprocessing to mine high-quality triplets where  
  negative examples are semantically hard and structurally unrelated

In [3]:
from utils.data_prep import save_article_titles

save_article_titles(WIKIDATA_JSONL_DIR, ARTICLE_TITLES_JSON)

Loading Article Titles From Files:   2%|▏         | 246/15843 [00:10<11:15, 23.09file/s]


KeyboardInterrupt: 

In [ ]:
from utils.page_links import save_embeddings_with_spark, create_ivfpq_faiss_index

model = SentenceTransformer(MODEL_NAME)
model.to('cuda')

if not os.path.exists(FAISS_PARA_META_PATH):
    save_embeddings_with_spark(
        WIKIDATA_JSON_DIR, 
        model, 
        EMBEDDINGS_DIR, 
        FAISS_PARA_META_PATH
    )
    
if not os.path.exists(FAISS_PARA_INDEX_PATH):
    create_ivfpq_faiss_index(
        EMBEDDINGS_DIR, 
        FAISS_PARA_INDEX_PATH,
        nlist=1024,      # Better clustering (was 100)
        nprobe=16,       # Better search accuracy (was 10)
        m=48,            # Better compression balance (was 16)
        nbits=8,         # Keep at 8
        train_size=100000,
        batch_size=1000
    )

Loading JSON into Spark DataFrame...


Splitting paragraphs...
Processing all paragraphs at once...


Encoding paragraphs:  92%|█████████▏| 61759109/66815288 [6:16:14<53:26, 1576.99it/s]  

In [ ]:
from utils.convert_faiss_meta_to_db import main

main()

In [ ]:
from utils.triplet_mining import mine_triplets_parallel_shared

FAISS_PARA_META_DB_PATH = r"../data/processed/faiss_index/paragraphs.index.meta.db"

total_triplets = mine_triplets_parallel_shared(
    wikidata_jsonl_dir=WIKIDATA_JSONL_DIR,
    faiss_index_path=FAISS_PARA_INDEX_PATH,
    faiss_meta_db_path=FAISS_PARA_META_DB_PATH,
    article_titles_path=ARTICLE_TITLES_JSON,
    link_graph_path=WIKI_LINK_GRAPH_DB_PATH,
    triplet_output_dir=TRIPLETS_DIR,
    num_processes=1,  # Reduced to 2 for GPU FAISS + models to fit in 8GB VRAM
    max_files_per_worker=200  # Increased since fewer processes
)

print(f"Triplet mining complete. Generated {total_triplets} triplets.")

Loading shared resources...
Loaded 4313403 article titles
Loading FAISS index...
Loaded 4313403 article titles
Loading FAISS index...


Gather total lines in triplets for model fit

In [ ]:
# Initialize Spark session
spark = SparkSession.builder \
        .appName("Capstone") \
        .master("local[*]") \
        .config("spark.driver.memory", "20g") \
        .config("spark.sql.shuffle.partitions", "100") \
        .config("spark.local.dir", "../spark-temp") \
        .config("spark.driver.maxResultSize", "2g") \
        .getOrCreate()

# Calculate total lines so that we can determine epoch size
total_lines = spark.read.json(f"{TRIPLETS_DIR}/*.jsonl", multiLine=False).count()
print("Total lines across training files:", total_lines)

# Stop spark
spark.stop()

25/06/28 04:05:01 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: ../data/processed/triplets/parallel_parts/*.jsonl.
java.io.FileNotFoundException: File ../data/processed/triplets/parallel_parts/*.jsonl does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:917)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1238)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:907)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.FileStreamSink$.hasMetadata(FileStreamSink.scala:56)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:381)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:1

AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/app/files/semantic-wiki-search/data/processed/triplets/parallel_parts/*.jsonl. SQLSTATE: 42K03

---

## Step 9: Train Embedding Model with Triplet Loss

- Loads mined triplets  
- Creates or loads a `TextVectorization` layer  
- Trains a custom Transformer encoder on the triplets using triplet loss  
- Saves the best model weights

In [ ]:
from keras.callbacks import EarlyStopping, ModelCheckpoint
from utils.custom_embedder import create_vectorizer, load_triplet_dataset_streamed, CustomEncoder, TripletTrainer

vocab_size = 30000
max_len = 32
embed_dim = 128
num_heads = 8
ff_dim = 256
batch_size = 512
num_epochs = 30
learning_rate = 1e-4
total_lines = 15123359

# Load or create vectorizer
if os.path.exists(VECTORIZER_DIR):
    print("Loading saved vectorizer")
    vectorizer = tf.keras.models.load_model(VECTORIZER_DIR)
else:
    vectorizer = create_vectorizer(TRIPLETS_DIR, VECTORIZER_DIR, vocab_size, max_len)

# Load dataset
train_dataset = load_triplet_dataset_streamed(TRIPLETS_DIR, vectorizer, batch_size)

# Model setup
encoder = CustomEncoder(vocab_size, max_len, embed_dim, num_heads, ff_dim)
model = TripletTrainer(encoder)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate))

# Callbacks
callbacks = [
    EarlyStopping(monitor="loss", patience=2),
    ModelCheckpoint(
        filepath=f"{WEIGHTS_DIR}/best_encoder.weights.h5",
        monitor="loss",
        save_best_only=True,
        save_weights_only=True
    )
]
# Training
model.fit(
    train_dataset.repeat(),  # infinite generator
    steps_per_epoch=total_lines // batch_size,
    epochs=num_epochs,
    callbacks=callbacks
)

print("Training complete and weights saved.")

---

## Step 10: Embed Anchors with Trained Encoder

- Extracts all unique anchor paragraphs  
- Vectorizes them with the trained encoder  
- Saves their embeddings in chunked `.npy` files

In [ ]:
vocab_size = 30000
max_len = 32
embed_dim = 128
num_heads = 8
ff_dim = 256
batch_size = 512
num_epochs = 30
learning_rate = 1e-4
total_lines = 15123359

encoder = CustomEncoder(vocab_size, max_len, embed_dim, num_heads, ff_dim, num_layers=2)
encoder.load_weights(f"{WEIGHTS_DIR}/best_encoder.weights.h5")

In [ ]:
anchors = set()
files = os.listdir(TRIPLETS_DIR)
for file in tqdm(files, desc="Loading Anchors"):
    if not file.endswith(".jsonl"):
        continue
    with open(os.path.join(TRIPLETS_DIR, file), "r", encoding="utf-8") as f:
        for line in f:
            triplet = json.loads(line)
            anchors.add(triplet["anchor"])

# Save to disk
os.makedirs(EMBEDDINGS_DIR, exist_ok=True)
with open(UNIQUE_ANCHORS_PATH, "w", encoding="utf-8") as out_f:
    for anchor in tqdm(sorted(anchors), desc="Saving Unique Anchors"):
        out_f.write(anchor + "\n")


Saving Unique Anchors: 100%|██████████| 15117457/15117457 [01:57<00:00, 128939.87it/s]


In [ ]:
def anchor_batch_generator(filepath, batch_size):
    with open(filepath, "r", encoding="utf-8") as f:
        batch = []
        for line in f:
            batch.append(line.strip())
            if len(batch) == batch_size:
                yield batch
                batch = []
        if batch:
            yield batch

In [ ]:
if os.path.exists(VECTORIZER_DIR):
    print("Loading saved vectorizer")
    vectorizer = tf.keras.models.load_model(VECTORIZER_DIR)
else:
    vectorizer = create_vectorizer(TRIPLETS_DIR, VECTORIZER_DIR, vocab_size, max_len)

batch_size = 4096
os.makedirs(EMBEDDINGS_CHUNK_PATH, exist_ok=True)
for i, anchor_batch in enumerate(tqdm(anchor_batch_generator(UNIQUE_ANCHORS_PATH, batch_size), total=len(anchors) // batch_size, desc="Encoding Anchors")):
    anchor_batch_tensor = tf.constant(anchor_batch)
    tokenized = vectorizer(anchor_batch_tensor)
    embeddings = encoder(tokenized, training=False).numpy()
    
    if i % 100 == 0 and i != 0:
        np.save(f"{EMBEDDINGS_CHUNK_PATH}/embeddings_chunk_{i//100}.npy", embeddings)
    if i == len(anchors) // batch_size:
        np.save(f"{EMBEDDINGS_CHUNK_PATH}/embeddings_chunk_{(i//100)+1}.npy", embeddings)

Fitting vectorizer on all data (streamed)


yielding text:  43%|████▎     | 7177/16657 [40:01<1:07:28,  2.34it/s]

---

## Step 11: Build FAISS Index

- Loads chunked embeddings from disk  
- Builds a FAISS index for efficient vector search  
- Saves the FAISS index to disk for later querying

In [ ]:
from utils.faiss_index import create_faiss_index_from_dir

# Creates embeddings and faiss.index
create_faiss_index_from_dir(EMBEDDINGS_CHUNK_PATH, FAISS_MODEL_PATH)

Found 37 embedding files. Building FAISS index...


Processing embedding files: 100%|██████████| 37/37 [00:00<00:00, 56.22it/s]


FAISS index saved.


---

## Step 12: Perform a Sample Query

- Loads the trained model and vectorizer  
- Embeds a sample query (e.g. "Who was involved in World War 2?")  
- Searches the FAISS index to retrieve top-10 matches

In [ ]:
vectorizer = tf.keras.models.load_model(VECTORIZER_DIR)

# Looad the weights
vocab_size = 30000
max_len = 32
embed_dim = 128
num_heads = 8
ff_dim = 256
batch_size = 512
num_epochs = 30
learning_rate = 1e-4

encoder = CustomEncoder(vocab_size, max_len, embed_dim, num_heads, ff_dim, num_layers=2)
encoder.load_weights(f"{WEIGHTS_DIR}/best_encoder.weights.h5")

# Your query
query = "Who was involved in World War 2?"

query_seq = vectorizer(tf.constant([query]))
query_embedding = encoder(query_seq).numpy()
query_embedding = query_embedding.astype(np.float32)

In [ ]:
from utils.faiss_index import query_faiss

indices = query_faiss(FAISS_MODEL_PATH, query_embedding, 10)

# Load article metadata
with open(ARTICLE_METADATA_PATH, encoding='utf-8') as f:
    metadata = json.load(f)

# Retrieve top-k articles
results = [metadata[i] for i in indices[0]]

---

## Step 13: Rerank with Cosine Similarity

- Retrieves paragraph embeddings from disk based on FAISS indices  
- Reranks results using cosine similarity to improve accuracy  
- Outputs the final list of results with title and URL

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

chunk_files = sorted([os.path.join(EMBEDDINGS_CHUNK_PATH, f) 
                      for f in os.listdir(EMBEDDINGS_CHUNK_PATH) 
                      if f.endswith(".npy")])
chunk_size = 4096

# Flatten FAISS indices into one list
faiss_indices = indices[0].tolist()

# Map: index -> (chunk_idx, local_offset)
index_map = {
    idx: (idx // chunk_size, idx % chunk_size)
    for idx in faiss_indices
}

# Load only the needed chunks
required_chunks = sorted(set(chunk_idx for chunk_idx, _ in index_map.values()))
chunk_data = {}

for chunk_idx in required_chunks:
    chunk_path = chunk_files[chunk_idx]
    chunk_data[chunk_idx] = np.load(chunk_path)

# Reconstruct article embeddings
article_embeddings = {
    idx: chunk_data[chunk_idx][offset]
    for idx, (chunk_idx, offset) in index_map.items()
}

# Load metadata aligned to the full dataset
with open(ARTICLE_METADATA_PATH, "r", encoding="utf-8") as f:
    metadata = json.load(f)

# Normalize query vector
query_vec = query_embedding[0].reshape(1, -1)

# Collect article dicts for reranking
top_articles = [
    metadata[i] | {"vec": article_embeddings[i]}
    for i in faiss_indices
]

# Rerank using cosine similarity
top_articles.sort(
    key=lambda x: cosine_similarity(query_vec, x["vec"].reshape(1, -1))[0][0],
    reverse=True
)

---

## Step 14: Evaluate Retrieval Performance

- Loads a test set of queries and ground-truth articles  
- Evaluates performance using metrics like:  
  - Top-K Accuracy  
  - Precision@K  
  - Recall@K  
  - Mean Reciprocal Rank (MRR)

In [ ]:
# Final top-k results
final_results = [
    {
        "title": article["title"],
        "url": article.get("url", "N/A")
    }
    for article in top_articles#[:5]
]

pprint.pprint(final_results)

[{'title': 'Huntly, New Zealand',
  'url': 'https://en.wikipedia.org/wiki?curid=264191'},
 {'title': 'Halfway, Oregon',
  'url': 'https://en.wikipedia.org/wiki?curid=130677'},
 {'title': 'Growth stock', 'url': 'https://en.wikipedia.org/wiki?curid=426130'},
 {'title': 'Polycystine', 'url': 'https://en.wikipedia.org/wiki?curid=320877'},
 {'title': 'Voices Carry (album)',
  'url': 'https://en.wikipedia.org/wiki?curid=1056233'},
 {'title': 'House, New Mexico',
  'url': 'https://en.wikipedia.org/wiki?curid=125952'},
 {'title': 'Red Hill, South Carolina',
  'url': 'https://en.wikipedia.org/wiki?curid=134536'},
 {'title': 'RKKY interaction',
  'url': 'https://en.wikipedia.org/wiki?curid=1227105'},
 {'title': 'Diplomatic mission',
  'url': 'https://en.wikipedia.org/wiki?curid=8970'},
 {'title': 'Elizabeth Proctor',
  'url': 'https://en.wikipedia.org/wiki?curid=1047314'}]


In [ ]:
from utils.top_k_testing import evaluate_all_metrics, retrieval_function, load_test_set

test_set = load_test_set(TEST_QUERIES_PATH)
results = evaluate_all_metrics(test_set, retrieval_function)

print("Evaluation Metrics:")
for metric, value in results.items():
    print(f"{metric}: {value}")

FileNotFoundError: [Errno 2] No such file or directory: '../data/test_data/test_queries.json'